# 01.5 — Stemming & Lemmatization

**Problem it solves:** 'run', 'runs', 'running', 'ran' all mean the same thing. Without normalization, a search for 'run' misses documents containing 'running'. Stemming and lemmatization map inflected forms to a common base.

**Stemming:** Crude, fast — chops suffixes with heuristic rules. May produce non-words ('studies' -> 'studi').
**Lemmatization:** Uses a vocabulary + morphological analysis to return the actual dictionary form ('studies' -> 'study').

---

## Part 1: Porter Stemmer From Scratch

The most widely-used English stemmer. Defined by 5 rule sets applied in sequence.

In [ ]:
import re

class PorterStemmer:
    """
    Simplified Porter Stemmer (major steps only).
    Full spec: https://tartarus.org/martin/PorterStemmer/def.txt
    """
    
    def _is_vowel(self, word: str, i: int) -> bool:
        if word[i] in 'aeiou':
            return True
        if word[i] == 'y' and i > 0 and word[i-1] not in 'aeiou':
            return True
        return False
    
    def _measure(self, stem: str) -> int:
        """Count VC sequences (consonant-vowel transitions).
        m=0: no VC, e.g. 'tr', 'ee'
        m=1: one VC, e.g. 'tree', 'trouble'
        Higher m = more 'substantial' stem, safer to modify.
        """
        m = 0
        in_vowel = False
        for i, c in enumerate(stem):
            v = self._is_vowel(stem, i)
            if v and not in_vowel:
                in_vowel = True
            elif not v and in_vowel:
                m += 1
                in_vowel = False
        return m
    
    def _has_vowel(self, stem: str) -> bool:
        return any(self._is_vowel(stem, i) for i in range(len(stem)))
    
    def _ends_double_consonant(self, word: str) -> bool:
        return (len(word) >= 2 and 
                word[-1] == word[-2] and 
                word[-1] not in 'aeiou')
    
    def _ends_cvc(self, word: str) -> bool:
        """Ends consonant-vowel-consonant where last consonant is not w, x, y."""
        if len(word) < 3:
            return False
        return (not self._is_vowel(word, -1) and
                word[-1] not in 'wxy' and
                self._is_vowel(word, -2) and
                not self._is_vowel(word, -3))
    
    def _step1a(self, word: str) -> str:
        """Plurals and past tense."""
        if word.endswith('sses'):
            return word[:-2]
        if word.endswith('ies'):
            return word[:-2]
        if word.endswith('ss'):
            return word
        if word.endswith('s'):
            return word[:-1]
        return word
    
    def _step1b(self, word: str) -> str:
        """Past tense and gerunds."""
        if word.endswith('eed'):
            stem = word[:-3]
            return word[:-1] if self._measure(stem) > 0 else word
        
        for suffix in ['ed', 'ing']:
            if word.endswith(suffix):
                stem = word[:-len(suffix)]
                if self._has_vowel(stem):
                    word = stem
                    # Post-processing
                    if word.endswith(('at', 'bl', 'iz')):
                        return word + 'e'
                    if self._ends_double_consonant(word) and word[-1] not in 'lsz':
                        return word[:-1]
                    if self._measure(word) == 1 and self._ends_cvc(word):
                        return word + 'e'
                    return word
        return word
    
    def _step1c(self, word: str) -> str:
        """y -> i when preceded by vowel."""
        if word.endswith('y') and self._has_vowel(word[:-1]):
            return word[:-1] + 'i'
        return word
    
    def _step2(self, word: str) -> str:
        """Derivational suffixes."""
        rules = [
            ('ational', 'ate'), ('tional', 'tion'), ('enci', 'ence'),
            ('anci', 'ance'), ('izer', 'ize'), ('iser', 'ize'),
            ('abli', 'able'), ('alli', 'al'), ('entli', 'ent'),
            ('eli', 'e'), ('ousli', 'ous'), ('ization', 'ize'),
            ('isation', 'ize'), ('ation', 'ate'), ('ator', 'ate'),
            ('alism', 'al'), ('iveness', 'ive'), ('fulness', 'ful'),
            ('ousness', 'ous'), ('aliti', 'al'), ('iviti', 'ive'),
            ('biliti', 'ble'),
        ]
        for suffix, replacement in rules:
            if word.endswith(suffix):
                stem = word[:-len(suffix)]
                if self._measure(stem) > 0:
                    return stem + replacement
        return word
    
    def _step3(self, word: str) -> str:
        """More derivational suffixes."""
        rules = [
            ('icate', 'ic'), ('ative', ''), ('alize', 'al'),
            ('iciti', 'ic'), ('ical', 'ic'), ('ful', ''), ('ness', ''),
        ]
        for suffix, replacement in rules:
            if word.endswith(suffix):
                stem = word[:-len(suffix)]
                if self._measure(stem) > 0:
                    return stem + replacement
        return word
    
    def _step4(self, word: str) -> str:
        """Removes final derivational suffixes if m > 1."""
        suffixes = ['al', 'ance', 'ence', 'er', 'ic', 'able', 'ible',
                    'ant', 'ement', 'ment', 'ent', 'ism', 'ate', 'iti',
                    'ous', 'ive', 'ize']
        for suffix in suffixes:
            if word.endswith(suffix):
                stem = word[:-len(suffix)]
                if self._measure(stem) > 1:
                    return stem
        if word.endswith('ion'):
            stem = word[:-3]
            if self._measure(stem) > 1 and stem and stem[-1] in 'st':
                return stem
        return word
    
    def _step5a(self, word: str) -> str:
        """Remove final e."""
        if word.endswith('e'):
            stem = word[:-1]
            if self._measure(stem) > 1:
                return stem
            if self._measure(stem) == 1 and not self._ends_cvc(stem):
                return stem
        return word
    
    def _step5b(self, word: str) -> str:
        """Remove double l."""
        if (self._measure(word) > 1 and 
            self._ends_double_consonant(word) and 
            word.endswith('l')):
            return word[:-1]
        return word
    
    def stem(self, word: str) -> str:
        word = word.lower()
        if len(word) <= 2:
            return word
        word = self._step1a(word)
        word = self._step1b(word)
        word = self._step1c(word)
        word = self._step2(word)
        word = self._step3(word)
        word = self._step4(word)
        word = self._step5a(word)
        word = self._step5b(word)
        return word

stemmer = PorterStemmer()

test_groups = {
    'run forms': ['run', 'runs', 'running', 'ran', 'runner'],
    'study forms': ['study', 'studies', 'studying', 'studied'],
    'connection forms': ['connect', 'connected', 'connecting', 'connection', 'connections', 'connective', 'connectivity'],
    'troublesome': ['goes', 'going', 'gone', 'went'],  # irregular — stemmer fails here
}

for group, words in test_groups.items():
    stems = [stemmer.stem(w) for w in words]
    print(f"\n{group}:")
    for w, s in zip(words, stems):
        print(f"  {w:20} -> {s}")

## Part 2: Stemmer Accuracy and Failure Modes

In [ ]:
# Overstemming: two DIFFERENT words -> same stem
overstem_pairs = [
    ('general', 'generalize'),   # both -> 'gener'
    ('experiment', 'experience'), # both -> 'experi'
    ('university', 'universe'),   # can overlap
]
print("=== OVERSTEMMING (false positives) ===")
for w1, w2 in overstem_pairs:
    s1, s2 = stemmer.stem(w1), stemmer.stem(w2)
    collision = '⚠️  COLLISION' if s1 == s2 else ''
    print(f"  {w1:20} -> {s1:15}  {w2:20} -> {s2:15} {collision}")

# Understemming: same root -> different stems
print("\n=== UNDERSTEMMING (false negatives) ===")
understem = [
    ('ran', 'run'),    # irregular past tense — stemmer won't unify these
    ('went', 'go'),
    ('better', 'good'),
    ('mice', 'mouse'),
]
for w1, w2 in understem:
    s1, s2 = stemmer.stem(w1), stemmer.stem(w2)
    unified = '✓' if s1 == s2 else '✗ NOT UNIFIED'
    print(f"  {w1:10} -> {s1:10}  {w2:10} -> {s2:10} {unified}")

## Part 3: Rule-Based Lemmatizer

Lemmatization needs a morphological dictionary. We'll build a small one to show the idea.

In [ ]:
class RuleBasedLemmatizer:
    """Demonstrates the structure of a lemmatizer.
    Real lemmatizers (spaCy, NLTK WordNet) have full lexicons."""
    
    # Irregular forms: {inflected: lemma}
    IRREGULAR_VERBS = {
        'ran': 'run', 'gone': 'go', 'went': 'go', 'goes': 'go',
        'was': 'be', 'were': 'be', 'is': 'be', 'are': 'be', 'been': 'be',
        'had': 'have', 'has': 'have',
        'did': 'do', 'does': 'do', 'done': 'do',
        'saw': 'see', 'seen': 'see',
        'knew': 'know', 'known': 'know',
        'took': 'take', 'taken': 'take',
        'said': 'say', 'says': 'say',
    }
    IRREGULAR_NOUNS = {
        'mice': 'mouse', 'men': 'man', 'women': 'woman',
        'children': 'child', 'teeth': 'tooth', 'feet': 'foot',
        'geese': 'goose', 'oxen': 'ox',
    }
    
    # Regular suffix rules: [(suffix, replacement, pos)]
    VERB_RULES = [
        ('ing', '', 'v'),     # running -> run (oversimplified)
        ('ied', 'y', 'v'),    # studied -> study
        ('ies', 'y', 'v'),    # studies -> study
        ('ed', '', 'v'),      # walked -> walk
        ('s', '', 'v'),       # runs -> run (last resort)
    ]
    NOUN_RULES = [
        ('sses', 'ss', 'n'),   # glasses -> glass
        ('ies', 'y', 'n'),     # babies -> baby
        ('ves', 'f', 'n'),     # leaves -> leaf
        ('s', '', 'n'),        # cats -> cat
    ]
    
    def lemmatize(self, word: str, pos: str = 'n') -> str:
        word = word.lower()
        
        # Check irregular forms first
        if pos == 'v' and word in self.IRREGULAR_VERBS:
            return self.IRREGULAR_VERBS[word]
        if pos == 'n' and word in self.IRREGULAR_NOUNS:
            return self.IRREGULAR_NOUNS[word]
        
        # Apply regular rules
        rules = self.VERB_RULES if pos == 'v' else self.NOUN_RULES
        for suffix, replacement, _ in rules:
            if word.endswith(suffix) and len(word) - len(suffix) >= 2:
                return word[:-len(suffix)] + replacement
        
        return word

lem = RuleBasedLemmatizer()

print("=== STEMMER vs LEMMATIZER ===")
print(f"{'Word':15} {'POS':5} {'Stem':15} {'Lemma':15}")
print('-' * 55)
test = [
    ('studies', 'n'), ('studies', 'v'), ('studying', 'v'),
    ('ran', 'v'), ('went', 'v'), ('better', 'a'),
    ('mice', 'n'), ('leaves', 'n'), ('babies', 'n'),
    ('walking', 'v'), ('walked', 'v'),
]
for word, pos in test:
    stem = stemmer.stem(word)
    lemma = lem.lemmatize(word, pos)
    print(f"{word:15} {pos:5} {stem:15} {lemma:15}")

## Part 4: When to Stem vs Lemmatize vs Neither

In [ ]:
# Decision guide with examples

scenarios = {
    "IR / Search": {
        "recommendation": "Stemming",
        "reason": "Speed matters, recall > precision. 'running shoes' query should hit 'run'",
        "caveat": "Avoid for technical domains where 'operation' != 'operational'"
    },
    "Text Classification": {
        "recommendation": "Lemmatization or neither",
        "reason": "Feature quality matters more than recall. Neural models handle morphology themselves.",
        "caveat": "For classical ML (bag-of-words), lemmatize. For neural, skip it."
    },
    "Language Modeling": {
        "recommendation": "Neither",
        "reason": "The model needs to see inflected forms to learn grammar and tense.",
        "caveat": "Subword tokenization (BPE) implicitly handles morphology"
    },
    "Keyword Extraction": {
        "recommendation": "Lemmatization",
        "reason": "Keywords should be in dictionary form for readability",
        "caveat": "Needs POS tagging to lemmatize correctly"
    },
}

for task, info in scenarios.items():
    print(f"\n{task}")
    print(f"  Recommendation: {info['recommendation']}")
    print(f"  Reason: {info['reason']}")
    print(f"  Caveat: {info['caveat']}")

## Summary

**Stemming:** Fast, aggressive, language-specific heuristics. Returns non-words. Use for search where recall matters.

**Lemmatization:** Slower, needs a lexicon + POS tags, returns real words. Use when word form matters.

**In 2024:** Subword tokenization largely replaces both for neural models. But for classical IR (BM25), stemming still helps recall significantly.

**What breaks it:**
- Irregular morphology (go/went, mouse/mice) — no rule system handles this, needs a lexicon
- Domain-specific terms (medical, legal) need domain lexicons
- Multilingual: Porter only works on English; each language needs its own rules

---
**Next:** 01.6 — Sentence Segmentation